# Journal Extension J7 — XAI / Grad-CAM + Ensemble Contribution Analysis
## Original 83.95% Thesis Protocol

This is the final major experimental notebook for the journal extension.

The frozen proposed method is an **ensemble**, so it does not have a single convolutional feature map. Therefore this notebook uses two complementary levels of explanation:

### A. Visual branch explanations
Grad-CAM is generated from the two foundational ResNet50 SSL branches:

- representative **SimCLR** model, selected using validation metrics only;
- standalone **Rotation SSL** fine-tuned model.

Grad-CAM target: the ground-truth class for the selected held-out test case.

Figures:
- one high-confidence correct example from each of the 8 classes;
- disagreement / final-rescue examples, emphasizing minority classes.

### B. Ensemble-level explanations
The saved logistic stacking model is analyzed directly:

- global source-block contribution magnitude;
- class-wise source coefficient magnitude;
- leave-one-source-out perturbation, when exact frozen reconstruction is verified;
- case-level source contributions toward the final predicted class;
- explicit reporting of the No-TTA anchor blend and class-bias vector.

This gives an interpretable explanation for a final method that is not itself a single CNN.

## Required Kaggle inputs

### 1. Original full thesis pipeline output — REQUIRED
Saved output of `last-part-of-the-previous-one`.

Required artifacts include:
- `rotation_ssl_paperlike/splits/rotation_fixed_train_val_test_split.json`
- `simclr_finetune_multiseed_bacc_v2/multiseed_results.csv`
- `simclr_finetune_multiseed_bacc_v2/seed_*/best_by_val_bacc.weights.h5`
- source-model validation/test probabilities
- `extended_imagenet_rotation_finetune/imagenet_rotation_finetune_best.weights.h5`

### 2. Confidence-gated final ensemble output — REQUIRED
Required:
- `reuse_only_all_model_ensemble/final_ensemble_config.json`
- `final_internal_probabilities.npz`
- `stacking_meta_model.joblib`

### 3. ISIC 2019 training images — REQUIRED
Attach the same ISIC 2019 image dataset used in the thesis.
The notebook maps images by `ISIC_xxxxxxx` filename stem.

### 4. J1 output — RECOMMENDED
If attached, J1 IDs are used for an additional order check.

## Compute
A Kaggle GPU is recommended for Grad-CAM, but no training is performed.
Internet is not required.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / CONFIG
# ============================================================
import os
import gc
import json
import math
import shutil
import zipfile
from collections import OrderedDict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j7_original_split_xai"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
AUDIT_DIR = OUTPUT_DIR / "audit"
CAM_DIR = OUTPUT_DIR / "gradcam"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, AUDIT_DIR, CAM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
NUM_CLASSES = 8
IMG_SIZE = 224

EXPECTED_VAL_N = 2533
EXPECTED_TEST_N = 2534

MAX_RESCUE_CASES = 8

print("=" * 96)
print("J7 — XAI / GRAD-CAM / ENSEMBLE CONTRIBUTION")
print("=" * 96)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))
print("Output:", OUTPUT_DIR)

J7 — XAI / GRAD-CAM / ENSEMBLE CONTRIBUTION
TensorFlow: 2.20.0
GPU available: True
Output: /kaggle/working/j7_original_split_xai


In [2]:
# ============================================================
# CELL 2 — GENERAL HELPERS
# ============================================================

def find_all(filename):
    return sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )

def normalize_probabilities(probabilities):
    p = np.asarray(probabilities, dtype=np.float64)

    if p.ndim != 2 or p.shape[1] != NUM_CLASSES:
        raise ValueError(
            f"Expected (N,{NUM_CLASSES}) probabilities, got {p.shape}"
        )

    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, 1e-12, None)
    p /= p.sum(axis=1, keepdims=True)

    return p.astype(np.float32)

def probability_columns(df):
    standard = [
        c for c in df.columns
        if c.startswith("prob_class_")
        and c.rsplit("_", 1)[-1].isdigit()
    ]

    if len(standard) == NUM_CLASSES:
        return sorted(
            standard,
            key=lambda c: int(c.rsplit("_", 1)[-1])
        )

    rotation = [
        c for c in df.columns
        if c.startswith("prob_")
        and len(c.split("_")) >= 2
        and c.split("_")[1].isdigit()
    ]

    if len(rotation) == NUM_CLASSES:
        return sorted(
            rotation,
            key=lambda c: int(c.split("_")[1])
        )

    exact = [
        f"prob_{i}"
        for i in range(NUM_CLASSES)
        if f"prob_{i}" in df.columns
    ]

    if len(exact) == NUM_CLASSES:
        return exact

    raise ValueError("Could not identify exactly 8 probability columns.")

def label_column(df):
    for c in ["y_true", "true_label", "label"]:
        if c in df.columns:
            return c
    raise ValueError("No label column found.")

def load_prediction_csv(path):
    df = pd.read_csv(path)
    y_col = label_column(df)
    p_cols = probability_columns(df)

    y = df[y_col].to_numpy(np.int32)
    p = normalize_probabilities(
        df[p_cols].to_numpy(np.float32)
    )

    return y, p

def temperature_scale(probabilities, temperature):
    p = normalize_probabilities(probabilities)

    logits = np.log(
        np.clip(p, 1e-12, 1.0)
    )

    logits /= float(temperature)
    logits -= logits.max(axis=1, keepdims=True)

    return normalize_probabilities(
        np.exp(logits)
    )

def apply_class_bias(probabilities, class_bias):
    p = normalize_probabilities(probabilities)

    logits = np.log(
        np.clip(p, 1e-12, 1.0)
    )

    logits += np.asarray(
        class_bias,
        dtype=np.float64,
    )[None, :]

    logits -= logits.max(axis=1, keepdims=True)

    return normalize_probabilities(
        np.exp(logits)
    )

def entropy(probabilities):
    p = np.clip(
        normalize_probabilities(probabilities),
        1e-12,
        1.0,
    )

    return -np.sum(
        p * np.log(p),
        axis=1,
        keepdims=True,
    )

print("Helpers ready.")

Helpers ready.


In [3]:
# ============================================================
# CELL 3 — LOCATE THE ORIGINAL THESIS PIPELINE
# ============================================================

split_hits = find_all(
    "rotation_fixed_train_val_test_split.json"
)

SPLIT_JSON = None

for p in split_hits:
    if (
        "rotation_ssl_paperlike" in str(p)
        and "splits" in str(p)
    ):
        SPLIT_JSON = p
        break

if SPLIT_JSON is None:
    raise FileNotFoundError(
        "Original thesis split JSON not found. "
        "Attach the saved `last-part-of-the-previous-one` output."
    )

# rotation_ssl_paperlike/splits/file -> rotation_ssl_paperlike
PIPELINE_ROOT = SPLIT_JSON.parent.parent

with open(SPLIT_JSON, "r") as f:
    SPLIT_INFO = json.load(f)

TEST_DF = pd.DataFrame(
    SPLIT_INFO["test"]
)

TEST_DF["filepath_original"] = TEST_DF["filepath"].astype(str)
TEST_DF["image_id"] = (
    TEST_DF["filepath_original"]
    .map(lambda x: Path(x).stem)
)
TEST_DF["label"] = TEST_DF["label"].astype(int)

if len(TEST_DF) != EXPECTED_TEST_N:
    raise RuntimeError(
        f"Wrong original test size: {len(TEST_DF)}"
    )

print("Pipeline root:")
print(PIPELINE_ROOT)

print("\nOriginal test rows:")
print(len(TEST_DF))

Pipeline root:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike

Original test rows:
2534


In [4]:
# ============================================================
# CELL 4 — MAP LOCKED TEST IDS TO THE ATTACHED ISIC IMAGES
# ============================================================

needed_ids = set(
    TEST_DF["image_id"].astype(str)
)

IMAGE_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

image_map = {}
duplicates = {}

print("Scanning Kaggle inputs for the 2,534 locked test images...")

for p in KAGGLE_INPUT.rglob("*"):
    if not p.is_file():
        continue

    if p.suffix.lower() not in IMAGE_EXTS:
        continue

    stem = p.stem

    if stem not in needed_ids:
        continue

    if stem not in image_map:
        image_map[stem] = str(p)
    else:
        duplicates.setdefault(
            stem,
            [image_map[stem]]
        )
        duplicates[stem].append(
            str(p)
        )

missing = sorted(
    needed_ids - set(image_map)
)

print("Required IDs :", len(needed_ids))
print("Matched      :", len(image_map))
print("Missing      :", len(missing))
print("Duplicate IDs:", len(duplicates))

if missing:
    raise FileNotFoundError(
        f"{len(missing)} original test images were not found. "
        f"Examples: {missing[:15]}. "
        "Attach the ISIC 2019 training image dataset."
    )

TEST_DF["filepath"] = (
    TEST_DF["image_id"]
    .astype(str)
    .map(image_map)
)

pd.DataFrame(
    [
        {
            "image_id": k,
            "candidate_paths": " || ".join(v),
        }
        for k, v in duplicates.items()
    ]
).to_csv(
    AUDIT_DIR / "duplicate_image_candidates.csv",
    index=False,
)

print("Image mapping: PASSED")

Scanning Kaggle inputs for the 2,534 locked test images...
Required IDs : 2534
Matched      : 2534
Missing      : 0
Duplicate IDs: 0
Image mapping: PASSED


In [5]:
# ============================================================
# CELL 5 — LOAD FROZEN FINAL CONFIG / PROBABILITIES / STACKER
# ============================================================

config_hits = find_all(
    "final_ensemble_config.json"
)

FINAL_CONFIG_JSON = None
FINAL_CONFIG = None

for p in sorted(
    config_hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else 9,
        len(p.parts),
        str(p),
    )
):
    try:
        with open(p, "r") as f:
            cfg = json.load(f)

        if (
            cfg.get("final_method")
            == "NoTTA_StackingBlend_ClassBias"
        ):
            FINAL_CONFIG_JSON = p
            FINAL_CONFIG = cfg
            break

    except Exception:
        continue

if FINAL_CONFIG_JSON is None:
    raise FileNotFoundError(
        "Frozen confidence-gated final config not found."
    )

final_npz_hits = find_all(
    "final_internal_probabilities.npz"
)

FINAL_NPZ = None

for p in sorted(
    final_npz_hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else 9,
        len(p.parts),
        str(p),
    )
):
    try:
        with np.load(p) as z:
            if not {
                "y_val",
                "y_test",
                "final_test_prob",
            }.issubset(z.files):
                continue

            if len(z["y_test"]) != EXPECTED_TEST_N:
                continue

        FINAL_NPZ = p
        break

    except Exception:
        continue

if FINAL_NPZ is None:
    raise FileNotFoundError(
        "Original frozen final probability file not found."
    )

with np.load(FINAL_NPZ) as z:
    Y_VAL = np.asarray(
        z["y_val"],
        dtype=np.int32,
    )

    Y_TEST = np.asarray(
        z["y_test"],
        dtype=np.int32,
    )

    FINAL_TEST_PROB = normalize_probabilities(
        z["final_test_prob"]
    )

if not np.array_equal(
    TEST_DF["label"].to_numpy(np.int32),
    Y_TEST,
):
    raise RuntimeError(
        "Original split JSON test labels do not align with final probabilities."
    )

stacker_hits = find_all(
    "stacking_meta_model.joblib"
)

STACKER_PATH = None

for p in sorted(
    stacker_hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else 9,
        len(p.parts),
        str(p),
    )
):
    STACKER_PATH = p
    break

if STACKER_PATH is None:
    raise FileNotFoundError(
        "stacking_meta_model.joblib not found."
    )

try:
    META_MODEL = joblib.load(
        STACKER_PATH
    )
    META_MODEL_LOAD_ERROR = None
except Exception as e:
    META_MODEL = None
    META_MODEL_LOAD_ERROR = repr(e)

print("Final config :", FINAL_CONFIG_JSON)
print("Final probs  :", FINAL_NPZ)
print("Meta-model   :", STACKER_PATH)
print("Meta load OK :", META_MODEL is not None)

Final config : /kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_ensemble_config.json
Final probs  : /kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_internal_probabilities.npz
Meta-model   : /kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/stacking_meta_model.joblib
Meta load OK : True


In [6]:
# ============================================================
# CELL 6 — LOAD THE SEVEN EXACT SOURCE PROBABILITY STREAMS
# ============================================================

MODEL_PATHS = OrderedDict([
    (
        "SimCLR_NoTTA",
        {
            "type": "npz",
            "path": (
                PIPELINE_ROOT
                / "simclr_finetune_multiseed_bacc_v2"
                / "multiseed_probabilities_no_tta.npz"
            ),
        },
    ),
    (
        "SimCLR_SelectedTTA",
        {
            "type": "npz",
            "path": (
                PIPELINE_ROOT
                / "simclr_finetune_multiseed_bacc_v2"
                / "tta_selection"
                / "selected_multiseed_tta_probabilities.npz"
            ),
        },
    ),
    (
        "Rotation",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "extended_imagenet_rotation_finetune"
                / "imagenet_rotation_val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "extended_imagenet_rotation_finetune"
                / "imagenet_rotation_test_predictions.csv"
            ),
        },
    ),
    (
        "FusionConcat",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "fusion_concat_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "fusion_concat_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv"
            ),
        },
    ),
    (
        "FusionGatedV2",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "fusion_gated_v2_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "fusion_gated_v2_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv"
            ),
        },
    ),
    (
        "FusionGatedV3",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "fusion_gated_v3_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "fusion_gated_v3_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv"
            ),
        },
    ),
    (
        "ResidualFusion",
        {
            "type": "npz",
            "path": (
                PIPELINE_ROOT
                / "residual_simclr_rotation_fusion_v1"
                / "multiseed_probabilities.npz"
            ),
        },
    ),
])

VAL_PROB = OrderedDict()
TEST_PROB = OrderedDict()

for model_name, spec in MODEL_PATHS.items():
    if spec["type"] == "npz":
        if not spec["path"].exists():
            raise FileNotFoundError(
                f"{model_name}: {spec['path']}"
            )

        with np.load(spec["path"]) as z:
            yv = np.asarray(
                z["y_val"],
                dtype=np.int32,
            )

            yt = np.asarray(
                z["y_test"],
                dtype=np.int32,
            )

            pv = normalize_probabilities(
                z["val_prob"]
            )

            pt = normalize_probabilities(
                z["test_prob"]
            )

    else:
        if not spec["val"].exists():
            raise FileNotFoundError(
                f"{model_name}: {spec['val']}"
            )

        if not spec["test"].exists():
            raise FileNotFoundError(
                f"{model_name}: {spec['test']}"
            )

        yv, pv = load_prediction_csv(
            spec["val"]
        )

        yt, pt = load_prediction_csv(
            spec["test"]
        )

    if not np.array_equal(
        yv,
        Y_VAL,
    ):
        raise RuntimeError(
            f"{model_name} validation labels are not aligned."
        )

    if not np.array_equal(
        yt,
        Y_TEST,
    ):
        raise RuntimeError(
            f"{model_name} test labels are not aligned."
        )

    VAL_PROB[model_name] = pv
    TEST_PROB[model_name] = pt

MODEL_NAMES = list(
    VAL_PROB.keys()
)

if MODEL_NAMES != list(
    FINAL_CONFIG["models"]
):
    raise RuntimeError(
        "Source-model order differs from frozen final config."
    )

print("Loaded source order:")
print(MODEL_NAMES)

Loaded source order:
['SimCLR_NoTTA', 'SimCLR_SelectedTTA', 'Rotation', 'FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']


In [7]:
# ============================================================
# CELL 7 — ENSEMBLE META-FEATURES / SOURCE IMPORTANCE
# ============================================================

TEMPERATURES = OrderedDict(
    (
        name,
        float(
            FINAL_CONFIG["temperatures"][name]
        )
    )
    for name in MODEL_NAMES
)

BEST_ALPHA = float(
    FINAL_CONFIG["stacking_alpha"]
)

ANCHOR_NAME = str(
    FINAL_CONFIG["anchor"]
)

CLASS_BIAS = np.asarray(
    FINAL_CONFIG["class_bias"],
    dtype=np.float64,
)

CALIBRATED_VAL = OrderedDict(
    (
        name,
        temperature_scale(
            VAL_PROB[name],
            TEMPERATURES[name],
        )
    )
    for name in MODEL_NAMES
)

CALIBRATED_TEST = OrderedDict(
    (
        name,
        temperature_scale(
            TEST_PROB[name],
            TEMPERATURES[name],
        )
    )
    for name in MODEL_NAMES
)

def build_meta_features(source_dictionary):
    blocks = []

    for name in MODEL_NAMES:
        p = source_dictionary[name]

        blocks.extend([
            p,
            np.log(
                np.clip(
                    p,
                    1e-12,
                    1.0,
                )
            ),
            np.max(
                p,
                axis=1,
                keepdims=True,
            ),
            entropy(p),
        ])

    return np.concatenate(
        blocks,
        axis=1,
    ).astype(np.float32)

META_X_VAL = build_meta_features(
    CALIBRATED_VAL
)

META_X_TEST = build_meta_features(
    CALIBRATED_TEST
)

META_BLOCK_SIZE = (
    NUM_CLASSES
    + NUM_CLASSES
    + 1
    + 1
)

if META_X_TEST.shape[1] != (
    len(MODEL_NAMES)
    * META_BLOCK_SIZE
):
    raise RuntimeError(
        "Unexpected meta-feature dimension."
    )

IMPORTANCE_AVAILABLE = (
    META_MODEL is not None
)

RECONSTRUCTION_AUDIT = {
    "meta_model_loaded": bool(
        IMPORTANCE_AVAILABLE
    ),
    "meta_model_load_error": META_MODEL_LOAD_ERROR,
}

if IMPORTANCE_AVAILABLE:
    scaler = META_MODEL.named_steps[
        "scaler"
    ]

    classifier = META_MODEL.named_steps[
        "classifier"
    ]

    X_TEST_SCALED = scaler.transform(
        META_X_TEST
    )

    coef = np.asarray(
        classifier.coef_,
        dtype=np.float64,
    )

    classes_seen = np.asarray(
        classifier.classes_,
        dtype=np.int32,
    )

    def align_probability(raw):
        aligned = np.zeros(
            (
                len(raw),
                NUM_CLASSES,
            ),
            dtype=np.float32,
        )

        for source_col, class_id in enumerate(
            classes_seen
        ):
            aligned[
                :,
                int(class_id),
            ] = raw[
                :,
                source_col,
            ]

        return normalize_probabilities(
            aligned
        )

    stack_prob = align_probability(
        META_MODEL.predict_proba(
            META_X_TEST
        )
    )

    blend_prob = normalize_probabilities(
        (
            1.0 - BEST_ALPHA
        )
        * CALIBRATED_TEST[
            ANCHOR_NAME
        ]
        + BEST_ALPHA
        * stack_prob
    )

    reconstructed_final = apply_class_bias(
        blend_prob,
        CLASS_BIAS,
    )

    reconstruction_mae = float(
        np.mean(
            np.abs(
                reconstructed_final
                - FINAL_TEST_PROB
            )
        )
    )

    reconstruction_argmax_match = float(
        np.mean(
            np.argmax(
                reconstructed_final,
                axis=1,
            )
            == np.argmax(
                FINAL_TEST_PROB,
                axis=1,
            )
        )
    )

    RECONSTRUCTION_AUDIT.update({
        "probability_mae_vs_saved_final":
            reconstruction_mae,

        "argmax_match_vs_saved_final":
            reconstruction_argmax_match,

        "exact_enough_for_leave_one_source_out":
            bool(
                reconstruction_mae
                < 1e-5
                and reconstruction_argmax_match
                > 0.999
            ),
    })

    global_rows = []

    for i, name in enumerate(
        MODEL_NAMES
    ):
        lo = i * META_BLOCK_SIZE
        hi = lo + META_BLOCK_SIZE

        # Additive standardized logit contribution.
        block_logits = (
            X_TEST_SCALED[:, lo:hi]
            @ coef[:, lo:hi].T
        )

        global_rows.append({
            "source_model": name,
            "mean_abs_meta_logit_contribution": float(
                np.mean(
                    np.abs(
                        block_logits
                    )
                )
            ),
            "coefficient_l2_norm": float(
                np.linalg.norm(
                    coef[:, lo:hi]
                )
            ),
        })

    GLOBAL_SOURCE_IMPORTANCE_DF = pd.DataFrame(
        global_rows
    )

    total = (
        GLOBAL_SOURCE_IMPORTANCE_DF[
            "mean_abs_meta_logit_contribution"
        ].sum()
    )

    GLOBAL_SOURCE_IMPORTANCE_DF[
        "normalized_meta_contribution"
    ] = (
        GLOBAL_SOURCE_IMPORTANCE_DF[
            "mean_abs_meta_logit_contribution"
        ]
        / max(
            total,
            1e-12,
        )
    )

    display(
        GLOBAL_SOURCE_IMPORTANCE_DF.sort_values(
            "normalized_meta_contribution",
            ascending=False,
        )
    )

    GLOBAL_SOURCE_IMPORTANCE_DF.to_csv(
        TABLE_DIR / "global_meta_source_importance.csv",
        index=False,
    )

else:
    print(
        "Saved meta-model could not be loaded; "
        "source-logit contribution analysis will be skipped."
    )

with open(
    AUDIT_DIR / "ensemble_reconstruction_audit.json",
    "w",
) as f:
    json.dump(
        RECONSTRUCTION_AUDIT,
        f,
        indent=2,
    )

print(json.dumps(RECONSTRUCTION_AUDIT, indent=2))

         source_model  mean_abs_meta_logit_contribution  coefficient_l2_norm  \
2            Rotation                          0.380007             1.010526   
4       FusionGatedV2                          0.243040             0.632161   
1  SimCLR_SelectedTTA                          0.242993             0.721692   
0        SimCLR_NoTTA                          0.241648             0.737833   
3        FusionConcat                          0.230962             0.611573   
5       FusionGatedV3                          0.212236             0.615610   
6      ResidualFusion                          0.209395             0.583600   

   normalized_meta_contribution  
2                      0.215879  
4                      0.138069  
1                      0.138042  
0                      0.137278  
3                      0.131207  
5                      0.120569  
6                      0.118956  

{
  "meta_model_loaded": true,
  "meta_model_load_error": null,
  "probability_mae_vs_saved_final": 1.685328786249496e-11,
  "argmax_match_vs_saved_final": 1.0,
  "exact_enough_for_leave_one_source_out": true
}


In [8]:
# ============================================================
# CELL 8 — CLASS-WISE META COEFFICIENT IMPORTANCE
# ============================================================

if IMPORTANCE_AVAILABLE:
    class_rows = []

    for class_row, class_id in enumerate(
        classes_seen
    ):
        for i, name in enumerate(
            MODEL_NAMES
        ):
            lo = i * META_BLOCK_SIZE
            hi = lo + META_BLOCK_SIZE

            class_rows.append({
                "class_id": int(class_id),
                "class_name": CLASS_NAMES[int(class_id)],
                "source_model": name,
                "coefficient_l2_norm": float(
                    np.linalg.norm(
                        coef[
                            class_row,
                            lo:hi,
                        ]
                    )
                ),
            })

    CLASS_SOURCE_IMPORTANCE_DF = pd.DataFrame(
        class_rows
    )

    display(
        CLASS_SOURCE_IMPORTANCE_DF
    )

    CLASS_SOURCE_IMPORTANCE_DF.to_csv(
        TABLE_DIR / "classwise_meta_source_importance.csv",
        index=False,
    )

    pivot = CLASS_SOURCE_IMPORTANCE_DF.pivot(
        index="class_name",
        columns="source_model",
        values="coefficient_l2_norm",
    )

    plt.figure(
        figsize=(10.0, 6.2)
    )

    plt.imshow(
        pivot.to_numpy(),
        interpolation="nearest",
    )

    plt.colorbar(
        label="Meta-classifier coefficient L2 norm"
    )

    plt.xticks(
        np.arange(
            len(pivot.columns)
        ),
        pivot.columns,
        rotation=40,
        ha="right",
    )

    plt.yticks(
        np.arange(
            len(pivot.index)
        ),
        pivot.index,
    )

    plt.title(
        "Class-wise Source Importance in the Logistic Stacker"
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "classwise_meta_source_importance.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
else:
    CLASS_SOURCE_IMPORTANCE_DF = pd.DataFrame()

    class_id class_name        source_model  coefficient_l2_norm
0          0        MEL        SimCLR_NoTTA             0.242814
1          0        MEL  SimCLR_SelectedTTA             0.251368
2          0        MEL            Rotation             0.302872
3          0        MEL        FusionConcat             0.222842
4          0        MEL       FusionGatedV2             0.215868
5          0        MEL       FusionGatedV3             0.137321
6          0        MEL      ResidualFusion             0.197756
7          1         NV        SimCLR_NoTTA             0.236300
8          1         NV  SimCLR_SelectedTTA             0.252142
9          1         NV            Rotation             0.287409
10         1         NV        FusionConcat             0.136272
11         1         NV       FusionGatedV2             0.186105
12         1         NV       FusionGatedV3             0.132415
13         1         NV      ResidualFusion             0.175852
14         2        BCC  

In [9]:
# ============================================================
# CELL 9 — LEAVE-ONE-SOURCE-OUT PERTURBATION
# ============================================================

LOSO_DF = pd.DataFrame()

if (
    IMPORTANCE_AVAILABLE
    and RECONSTRUCTION_AUDIT.get(
        "exact_enough_for_leave_one_source_out",
        False,
    )
):
    base_pred = np.argmax(
        reconstructed_final,
        axis=1,
    )

    base_bacc = float(
        balanced_accuracy_score(
            Y_TEST,
            base_pred,
        )
    )

    base_acc = float(
        accuracy_score(
            Y_TEST,
            base_pred,
        )
    )

    loso_rows = []

    for i, name in enumerate(
        MODEL_NAMES
    ):
        lo = i * META_BLOCK_SIZE
        hi = lo + META_BLOCK_SIZE

        # Set standardized block to zero = replace raw block
        # with the validation scaler mean.
        x_mod = X_TEST_SCALED.copy()
        x_mod[:, lo:hi] = 0.0

        stack_mod_raw = classifier.predict_proba(
            x_mod
        )

        stack_mod = align_probability(
            stack_mod_raw
        )

        blend_mod = normalize_probabilities(
            (
                1.0 - BEST_ALPHA
            )
            * CALIBRATED_TEST[
                ANCHOR_NAME
            ]
            + BEST_ALPHA
            * stack_mod
        )

        final_mod = apply_class_bias(
            blend_mod,
            CLASS_BIAS,
        )

        pred_mod = np.argmax(
            final_mod,
            axis=1,
        )

        mod_bacc = float(
            balanced_accuracy_score(
                Y_TEST,
                pred_mod,
            )
        )

        mod_acc = float(
            accuracy_score(
                Y_TEST,
                pred_mod,
            )
        )

        loso_rows.append({
            "removed_source": name,
            "baseline_bacc": base_bacc,
            "without_source_bacc": mod_bacc,
            "bacc_drop_when_removed": base_bacc - mod_bacc,
            "baseline_accuracy": base_acc,
            "without_source_accuracy": mod_acc,
            "accuracy_drop_when_removed": base_acc - mod_acc,
            "prediction_flips": int(
                np.sum(
                    pred_mod != base_pred
                )
            ),
            "mean_probability_shift": float(
                np.mean(
                    np.abs(
                        final_mod
                        - reconstructed_final
                    )
                )
            ),
        })

    LOSO_DF = pd.DataFrame(
        loso_rows
    ).sort_values(
        "bacc_drop_when_removed",
        ascending=False,
    )

    display(
        LOSO_DF
    )

    LOSO_DF.to_csv(
        TABLE_DIR / "leave_one_source_out_ensemble_importance.csv",
        index=False,
    )

    plt.figure(
        figsize=(9.0, 5.4)
    )

    plt.bar(
        LOSO_DF["removed_source"],
        100
        * LOSO_DF[
            "bacc_drop_when_removed"
        ],
    )

    plt.xticks(
        rotation=40,
        ha="right",
    )

    plt.ylabel(
        "BACC drop when source is removed (percentage points)"
    )

    plt.title(
        "Leave-One-Source-Out Importance of the Frozen Ensemble"
    )

    plt.axhline(
        0.0,
        linewidth=1,
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "leave_one_source_out_bacc_drop.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

else:
    print(
        "Leave-one-source-out analysis skipped because "
        "the saved final ensemble could not be reconstructed "
        "to the strict required tolerance."
    )

       removed_source  baseline_bacc  without_source_bacc  \
2            Rotation       0.839549             0.816729   
4       FusionGatedV2       0.839549             0.831437   
3        FusionConcat       0.839549             0.832028   
0        SimCLR_NoTTA       0.839549             0.832754   
6      ResidualFusion       0.839549             0.834246   
1  SimCLR_SelectedTTA       0.839549             0.837429   
5       FusionGatedV3       0.839549             0.837836   

   bacc_drop_when_removed  baseline_accuracy  without_source_accuracy  \
2                0.022821            0.83899                 0.840174   
4                0.008112            0.83899                 0.837806   
3                0.007521            0.83899                 0.838595   
0                0.006795            0.83899                 0.840568   
6                0.005303            0.83899                 0.837806   
1                0.002120            0.83899                 0.842541   


In [10]:
# ============================================================
# CELL 10 — SELECT JOURNAL XAI CASES
# ============================================================

FINAL_PRED = np.argmax(
    FINAL_TEST_PROB,
    axis=1,
)

FINAL_CONF = np.max(
    FINAL_TEST_PROB,
    axis=1,
)

SIMCLR_PRED = np.argmax(
    TEST_PROB["SimCLR_NoTTA"],
    axis=1,
)

ROTATION_PRED = np.argmax(
    TEST_PROB["Rotation"],
    axis=1,
)

# One high-confidence final-correct case per class.
correct_case_indices = []

for c in range(NUM_CLASSES):
    candidates = np.flatnonzero(
        (Y_TEST == c)
        & (FINAL_PRED == c)
    )

    if len(candidates) == 0:
        continue

    best = candidates[
        np.argmax(
            FINAL_CONF[candidates]
        )
    ]

    correct_case_indices.append(
        int(best)
    )

# Rescue/disagreement cases.
# Prioritize minority / clinically difficult classes.
priority_classes = [
    3,  # AK
    5,  # DF
    6,  # VASC
    7,  # SCC
    0,  # MEL
    2,  # BCC
    4,  # BKL
    1,  # NV
]

rescue_case_indices = []
used = set(
    correct_case_indices
)

for c in priority_classes:
    candidates = np.flatnonzero(
        (Y_TEST == c)
        & (FINAL_PRED == Y_TEST)
        & (
            (SIMCLR_PRED != Y_TEST)
            | (ROTATION_PRED != Y_TEST)
        )
    )

    candidates = [
        int(i)
        for i in candidates
        if int(i) not in used
    ]

    if not candidates:
        continue

    candidates = sorted(
        candidates,
        key=lambda i: FINAL_CONF[i],
        reverse=True,
    )

    chosen = candidates[0]
    rescue_case_indices.append(
        chosen
    )
    used.add(chosen)

    if len(rescue_case_indices) >= MAX_RESCUE_CASES:
        break

# Fill remaining rescue slots globally if necessary.
if len(rescue_case_indices) < MAX_RESCUE_CASES:
    candidates = np.flatnonzero(
        (FINAL_PRED == Y_TEST)
        & (
            (SIMCLR_PRED != Y_TEST)
            | (ROTATION_PRED != Y_TEST)
        )
    )

    candidates = [
        int(i)
        for i in candidates
        if int(i) not in used
    ]

    candidates = sorted(
        candidates,
        key=lambda i: FINAL_CONF[i],
        reverse=True,
    )

    for i in candidates:
        rescue_case_indices.append(
            i
        )
        used.add(i)

        if len(rescue_case_indices) >= MAX_RESCUE_CASES:
            break

def case_row(idx, category):
    return {
        "test_index": int(idx),
        "category": category,
        "image_id": str(
            TEST_DF.iloc[idx]["image_id"]
        ),
        "filepath": str(
            TEST_DF.iloc[idx]["filepath"]
        ),
        "true_class": CLASS_NAMES[
            int(Y_TEST[idx])
        ],
        "final_pred": CLASS_NAMES[
            int(FINAL_PRED[idx])
        ],
        "final_confidence": float(
            FINAL_CONF[idx]
        ),
        "simclr_multiseed_pred": CLASS_NAMES[
            int(SIMCLR_PRED[idx])
        ],
        "rotation_pred": CLASS_NAMES[
            int(ROTATION_PRED[idx])
        ],
        "final_rescues_simclr": bool(
            FINAL_PRED[idx] == Y_TEST[idx]
            and SIMCLR_PRED[idx] != Y_TEST[idx]
        ),
        "final_rescues_rotation": bool(
            FINAL_PRED[idx] == Y_TEST[idx]
            and ROTATION_PRED[idx] != Y_TEST[idx]
        ),
    }

CASE_DF = pd.DataFrame(
    [
        case_row(i, "correct_per_class")
        for i in correct_case_indices
    ]
    +
    [
        case_row(i, "final_rescue_or_disagreement")
        for i in rescue_case_indices
    ]
)

display(
    CASE_DF
)

CASE_DF.to_csv(
    TABLE_DIR / "selected_xai_cases.csv",
    index=False,
)

print(
    "Correct-per-class cases:",
    len(correct_case_indices)
)

print(
    "Rescue/disagreement cases:",
    len(rescue_case_indices)
)

    test_index                      category           image_id  \
0         1141             correct_per_class  ISIC_Resize_17633   
1          561             correct_per_class   ISIC_Resize_8610   
2          183             correct_per_class  ISIC_Resize_14728   
3         1008             correct_per_class  ISIC_Resize_22664   
4          876             correct_per_class   ISIC_Resize_2340   
5         1273             correct_per_class  ISIC_Resize_14277   
6          894             correct_per_class  ISIC_Resize_20602   
7          749             correct_per_class  ISIC_Resize_15627   
8           53  final_rescue_or_disagreement  ISIC_Resize_23499   
9         1441  final_rescue_or_disagreement  ISIC_Resize_19742   
10        1904  final_rescue_or_disagreement  ISIC_Resize_14268   
11        2370  final_rescue_or_disagreement  ISIC_Resize_18803   
12        2496  final_rescue_or_disagreement  ISIC_Resize_14069   
13        2357  final_rescue_or_disagreement  ISIC_Resize_1804

Correct-per-class cases: 8
Rescue/disagreement cases: 8


In [11]:
# ============================================================
# CELL 11 — BUILD THE EXACT REPRESENTATIVE SIMCLR MODEL
# ============================================================

SIMCLR_DIR = (
    PIPELINE_ROOT
    / "simclr_finetune_multiseed_bacc_v2"
)

SIMCLR_RESULTS_CSV = (
    SIMCLR_DIR
    / "multiseed_results.csv"
)

if not SIMCLR_RESULTS_CSV.exists():
    raise FileNotFoundError(
        str(SIMCLR_RESULTS_CSV)
    )

sim_results = pd.read_csv(
    SIMCLR_RESULTS_CSV
)

required_cols = {
    "seed",
    "val_bacc",
    "val_macro_f1",
    "val_accuracy",
}

if not required_cols.issubset(
    sim_results.columns
):
    raise ValueError(
        "SimCLR multi-seed results file is missing validation ranking columns."
    )

sim_ranking = sim_results.sort_values(
    [
        "val_bacc",
        "val_macro_f1",
        "val_accuracy",
        "seed",
    ],
    ascending=[
        False,
        False,
        False,
        True,
    ],
).reset_index(drop=True)

SIMCLR_REP_SEED = int(
    sim_ranking.iloc[0]["seed"]
)

SIMCLR_REP_WEIGHTS = (
    SIMCLR_DIR
    / f"seed_{SIMCLR_REP_SEED}"
    / "best_by_val_bacc.weights.h5"
)

if not SIMCLR_REP_WEIGHTS.exists():
    raise FileNotFoundError(
        str(SIMCLR_REP_WEIGHTS)
    )

def simclr_preprocess_for_resnet_tensor(img_batch):
    img_batch = tf.cast(
        img_batch,
        tf.float32,
    ) * 255.0

    return tf.keras.applications.resnet50.preprocess_input(
        img_batch
    )

def build_simclr_encoder(img_size=224):
    inputs = keras.Input(
        shape=(img_size, img_size, 3),
        name="simclr_input",
    )

    backbone = ResNet50(
        include_top=False,
        weights=None,
        input_shape=(
            img_size,
            img_size,
            3,
        ),
        pooling="avg",
    )

    x = backbone(
        inputs,
        training=False,
    )

    return keras.Model(
        inputs,
        x,
        name="simclr_resnet50_encoder",
    )

def build_simclr_model():
    encoder = build_simclr_encoder(
        IMG_SIZE
    )

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="finetune_img",
    )

    x = layers.Lambda(
        simclr_preprocess_for_resnet_tensor,
        name="resnet_preprocess",
    )(inputs)

    x = encoder(
        x,
        training=False,
    )

    x = layers.Dropout(
        0.30,
        name="finetune_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="finetune_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="simclr_finetune_model",
    )

    return model, encoder

tf.keras.backend.clear_session()

SIMCLR_MODEL, SIMCLR_ENCODER = (
    build_simclr_model()
)

SIMCLR_MODEL.load_weights(
    str(SIMCLR_REP_WEIGHTS)
)

print(
    "Representative SimCLR seed selected by validation only:",
    SIMCLR_REP_SEED
)

print(
    "Weights:",
    SIMCLR_REP_WEIGHTS
)

I0000 00:00:1786629182.129490      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Representative SimCLR seed selected by validation only: 62
Weights: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/simclr_finetune_multiseed_bacc_v2/seed_62/best_by_val_bacc.weights.h5


In [12]:
# ============================================================
# CELL 12 — BUILD THE EXACT ROTATION FINE-TUNED MODEL
# ============================================================

ROTATION_DIR = (
    PIPELINE_ROOT
    / "extended_imagenet_rotation_finetune"
)

ROTATION_WEIGHTS = (
    ROTATION_DIR
    / "imagenet_rotation_finetune_best.weights.h5"
)

if not ROTATION_WEIGHTS.exists():
    raise FileNotFoundError(
        str(ROTATION_WEIGHTS)
    )

def rotation_preprocess_for_resnet_tensor(img_batch):
    img_batch = (
        tf.cast(
            img_batch,
            tf.float32,
        )
        * 255.0
    )

    return tf.keras.applications.resnet50.preprocess_input(
        img_batch
    )

def build_rotation_encoder(img_size=224):
    inputs = keras.Input(
        shape=(
            img_size,
            img_size,
            3,
        ),
        name="rotation_ft_encoder_input",
    )

    backbone = ResNet50(
        include_top=False,
        weights=None,
        input_shape=(
            img_size,
            img_size,
            3,
        ),
        pooling="avg",
    )

    features = backbone(
        inputs,
        training=True,
    )

    return keras.Model(
        inputs,
        features,
        name="rotation_ft_resnet50_encoder",
    )

def build_rotation_model():
    encoder = build_rotation_encoder(
        IMG_SIZE
    )

    encoder.trainable = True

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="imagenet_rotation_input",
    )

    x = layers.Lambda(
        rotation_preprocess_for_resnet_tensor,
        name="resnet50_preprocess",
    )(inputs)

    x = encoder(
        x,
        training=True,
    )

    x = layers.Dropout(
        0.30,
        name="imagenet_rotation_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="imagenet_rotation_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="imagenet_rotation_finetune_model",
    )

    return model, encoder

ROTATION_MODEL, ROTATION_ENCODER = (
    build_rotation_model()
)

ROTATION_MODEL.load_weights(
    str(ROTATION_WEIGHTS)
)

print(
    "Rotation weights:",
    ROTATION_WEIGHTS
)

Rotation weights: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/extended_imagenet_rotation_finetune/imagenet_rotation_finetune_best.weights.h5


In [13]:
# ============================================================
# CELL 13 — GRAD-CAM HELPERS
# ============================================================

def find_resnet_backbone(
    encoder,
):
    candidates = [
        layer
        for layer in encoder.layers
        if isinstance(
            layer,
            keras.Model,
        )
    ]

    if not candidates:
        raise RuntimeError(
            f"No nested backbone model found in {encoder.name}."
        )

    # Prefer the nested ResNet50 model.
    candidates = sorted(
        candidates,
        key=lambda layer: (
            0
            if "resnet50" in layer.name.lower()
            or layer.name.lower() == "resnet50"
            else 1,
            layer.name,
        ),
    )

    return candidates[0]

SIMCLR_BACKBONE = find_resnet_backbone(
    SIMCLR_ENCODER
)

ROTATION_BACKBONE = find_resnet_backbone(
    ROTATION_ENCODER
)

LAST_CONV_LAYER = "conv5_block3_out"

if LAST_CONV_LAYER not in [
    layer.name
    for layer in SIMCLR_BACKBONE.layers
]:
    raise RuntimeError(
        f"{LAST_CONV_LAYER} not found in SimCLR ResNet50."
    )

if LAST_CONV_LAYER not in [
    layer.name
    for layer in ROTATION_BACKBONE.layers
]:
    raise RuntimeError(
        f"{LAST_CONV_LAYER} not found in Rotation ResNet50."
    )

SIMCLR_CONV_MODEL = keras.Model(
    SIMCLR_BACKBONE.input,
    SIMCLR_BACKBONE.get_layer(
        LAST_CONV_LAYER
    ).output,
)

ROTATION_CONV_MODEL = keras.Model(
    ROTATION_BACKBONE.input,
    ROTATION_BACKBONE.get_layer(
        LAST_CONV_LAYER
    ).output,
)

SIMCLR_CLASSIFIER = (
    SIMCLR_MODEL.get_layer(
        "finetune_classifier"
    )
)

ROTATION_CLASSIFIER = (
    ROTATION_MODEL.get_layer(
        "imagenet_rotation_classifier"
    )
)

def load_image_01(path):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im = im.resize(
            (
                IMG_SIZE,
                IMG_SIZE,
            ),
            resample=Image.Resampling.BICUBIC,
        )

        arr = (
            np.asarray(
                im,
                dtype=np.float32,
            )
            / 255.0
        )

    return arr

def gradcam_from_branch(
    image_01,
    branch,
    target_class,
):
    image_tensor = tf.convert_to_tensor(
        image_01[None, ...],
        dtype=tf.float32,
    )

    if branch == "SimCLR":
        preprocess = (
            simclr_preprocess_for_resnet_tensor
        )
        conv_model = (
            SIMCLR_CONV_MODEL
        )
        classifier = (
            SIMCLR_CLASSIFIER
        )
        backbone_training = False

    elif branch == "Rotation":
        preprocess = (
            rotation_preprocess_for_resnet_tensor
        )
        conv_model = (
            ROTATION_CONV_MODEL
        )
        classifier = (
            ROTATION_CLASSIFIER
        )
        backbone_training = True

    else:
        raise ValueError(
            branch
        )

    processed = preprocess(
        image_tensor
    )

    with tf.GradientTape() as tape:
        conv = conv_model(
            processed,
            training=backbone_training,
        )

        tape.watch(
            conv
        )

        # ResNet50(include_top=False, pooling='avg') is
        # global average pooling of conv5_block3_out.
        pooled = tf.reduce_mean(
            conv,
            axis=(1, 2),
        )

        logits = (
            tf.linalg.matmul(
                pooled,
                classifier.kernel,
            )
            + classifier.bias
        )

        target_score = logits[
            :,
            int(target_class),
        ]

    grads = tape.gradient(
        target_score,
        conv,
    )

    if grads is None:
        raise RuntimeError(
            f"Gradients are None for {branch}."
        )

    channel_weights = tf.reduce_mean(
        grads,
        axis=(1, 2),
    )

    cam = tf.reduce_sum(
        conv
        * channel_weights[
            :,
            None,
            None,
            :
        ],
        axis=-1,
    )

    cam = tf.nn.relu(
        cam
    )[0]

    cam_min = tf.reduce_min(
        cam
    )

    cam_max = tf.reduce_max(
        cam
    )

    cam = (
        cam
        - cam_min
    ) / (
        cam_max
        - cam_min
        + 1e-12
    )

    cam = tf.image.resize(
        cam[
            ...,
            None
        ],
        (
            IMG_SIZE,
            IMG_SIZE,
        ),
        method="bilinear",
    )[
        ...,
        0
    ]

    # Branch prediction from the exact reconstructed model.
    if branch == "SimCLR":
        branch_prob = SIMCLR_MODEL(
            image_tensor,
            training=False,
        ).numpy()[0]
    else:
        branch_prob = ROTATION_MODEL(
            image_tensor,
            training=False,
        ).numpy()[0]

    return (
        cam.numpy(),
        branch_prob,
    )

def overlay_cam(
    image_01,
    cam,
    alpha=0.42,
):
    cmap = plt.get_cmap(
        "jet"
    )

    heat = cmap(
        np.clip(
            cam,
            0.0,
            1.0,
        )
    )[
        ...,
        :3
    ]

    overlay = (
        (1.0 - alpha)
        * image_01
        + alpha
        * heat
    )

    return np.clip(
        overlay,
        0.0,
        1.0,
    )

print(
    "SimCLR backbone:",
    SIMCLR_BACKBONE.name
)

print(
    "Rotation backbone:",
    ROTATION_BACKBONE.name
)

print(
    "Grad-CAM layer:",
    LAST_CONV_LAYER
)

SimCLR backbone: resnet50
Rotation backbone: resnet50
Grad-CAM layer: conv5_block3_out


In [14]:
# ============================================================
# CELL 14 — GENERATE JOURNAL GRAD-CAM GRIDS
# ============================================================

def make_gradcam_grid(
    indices,
    title,
    output_filename,
):
    if len(indices) == 0:
        print(
            "No cases for",
            title
        )
        return pd.DataFrame()

    n_rows = len(indices)

    fig, axes = plt.subplots(
        n_rows,
        3,
        figsize=(
            11.0,
            3.1 * n_rows,
        ),
        squeeze=False,
    )

    rows = []

    for row_idx, test_idx in enumerate(
        indices
    ):
        record = TEST_DF.iloc[
            test_idx
        ]

        image = load_image_01(
            record["filepath"]
        )

        true_class = int(
            Y_TEST[test_idx]
        )

        sim_cam, sim_prob = (
            gradcam_from_branch(
                image,
                "SimCLR",
                true_class,
            )
        )

        rot_cam, rot_prob = (
            gradcam_from_branch(
                image,
                "Rotation",
                true_class,
            )
        )

        sim_overlay = overlay_cam(
            image,
            sim_cam,
        )

        rot_overlay = overlay_cam(
            image,
            rot_cam,
        )

        sim_pred = int(
            np.argmax(
                sim_prob
            )
        )

        rot_pred = int(
            np.argmax(
                rot_prob
            )
        )

        axes[
            row_idx,
            0,
        ].imshow(
            image
        )

        axes[
            row_idx,
            0,
        ].set_title(
            (
                f"{record['image_id']}\n"
                f"True={CLASS_NAMES[true_class]} | "
                f"Final={CLASS_NAMES[int(FINAL_PRED[test_idx])]}"
            ),
            fontsize=9,
        )

        axes[
            row_idx,
            1,
        ].imshow(
            sim_overlay
        )

        axes[
            row_idx,
            1,
        ].set_title(
            (
                f"SimCLR CAM → true class\n"
                f"rep pred={CLASS_NAMES[sim_pred]} "
                f"({sim_prob[sim_pred]:.2f})"
            ),
            fontsize=9,
        )

        axes[
            row_idx,
            2,
        ].imshow(
            rot_overlay
        )

        axes[
            row_idx,
            2,
        ].set_title(
            (
                f"Rotation CAM → true class\n"
                f"pred={CLASS_NAMES[rot_pred]} "
                f"({rot_prob[rot_pred]:.2f})"
            ),
            fontsize=9,
        )

        for col in range(3):
            axes[
                row_idx,
                col,
            ].axis(
                "off"
            )

        rows.append({
            "test_index": int(
                test_idx
            ),
            "image_id": str(
                record["image_id"]
            ),
            "true_class": CLASS_NAMES[
                true_class
            ],
            "final_pred": CLASS_NAMES[
                int(
                    FINAL_PRED[
                        test_idx
                    ]
                )
            ],
            "simclr_representative_pred": CLASS_NAMES[
                sim_pred
            ],
            "simclr_representative_confidence": float(
                sim_prob[
                    sim_pred
                ]
            ),
            "rotation_gradcam_model_pred": CLASS_NAMES[
                rot_pred
            ],
            "rotation_gradcam_model_confidence": float(
                rot_prob[
                    rot_pred
                ]
            ),
        })

    fig.suptitle(
        title,
        fontsize=14,
        y=1.002,
    )

    plt.tight_layout()

    path = (
        FIG_DIR
        / output_filename
    )

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    return pd.DataFrame(
        rows
    )

CORRECT_CAM_DF = make_gradcam_grid(
    correct_case_indices,
    (
        "Grad-CAM: One High-Confidence Correct Case per Class "
        "(Target = Ground-Truth Class)"
    ),
    "gradcam_correct_per_class.png",
)

RESCUE_CAM_DF = make_gradcam_grid(
    rescue_case_indices,
    (
        "Grad-CAM: Final-Model Rescue / Branch-Disagreement Cases "
        "(Target = Ground-Truth Class)"
    ),
    "gradcam_final_rescue_cases.png",
)

CORRECT_CAM_DF.to_csv(
    TABLE_DIR / "gradcam_correct_case_predictions.csv",
    index=False,
)

RESCUE_CAM_DF.to_csv(
    TABLE_DIR / "gradcam_rescue_case_predictions.csv",
    index=False,
)

In [15]:
# ============================================================
# CELL 15 — CASE-LEVEL META-SOURCE CONTRIBUTIONS
# ============================================================

CASE_CONTRIBUTION_DF = pd.DataFrame()

if IMPORTANCE_AVAILABLE:
    chosen_case_indices = (
        correct_case_indices
        + rescue_case_indices
    )

    contribution_rows = []

    final_saved_pred = np.argmax(
        FINAL_TEST_PROB,
        axis=1,
    )

    class_to_row = {
        int(class_id): row_idx
        for row_idx, class_id
        in enumerate(
            classes_seen
        )
    }

    for idx in chosen_case_indices:
        target_class = int(
            final_saved_pred[
                idx
            ]
        )

        if target_class not in class_to_row:
            continue

        coef_row = class_to_row[
            target_class
        ]

        for i, name in enumerate(
            MODEL_NAMES
        ):
            lo = i * META_BLOCK_SIZE
            hi = lo + META_BLOCK_SIZE

            contribution = float(
                np.dot(
                    X_TEST_SCALED[
                        idx,
                        lo:hi,
                    ],
                    coef[
                        coef_row,
                        lo:hi,
                    ],
                )
            )

            contribution_rows.append({
                "test_index": int(idx),
                "image_id": str(
                    TEST_DF.iloc[
                        idx
                    ][
                        "image_id"
                    ]
                ),
                "true_class": CLASS_NAMES[
                    int(
                        Y_TEST[
                            idx
                        ]
                    )
                ],
                "final_predicted_class": CLASS_NAMES[
                    target_class
                ],
                "source_model": name,
                "meta_logit_contribution_to_final_predicted_class": contribution,
            })

    CASE_CONTRIBUTION_DF = pd.DataFrame(
        contribution_rows
    )

    CASE_CONTRIBUTION_DF.to_csv(
        TABLE_DIR / "case_level_meta_source_contributions.csv",
        index=False,
    )

    pivot = CASE_CONTRIBUTION_DF.pivot(
        index="image_id",
        columns="source_model",
        values="meta_logit_contribution_to_final_predicted_class",
    )

    plt.figure(
        figsize=(
            10.5,
            max(
                5.0,
                0.45
                * len(pivot.index)
                + 2.0,
            ),
        )
    )

    max_abs = float(
        np.max(
            np.abs(
                pivot.to_numpy()
            )
        )
    )

    plt.imshow(
        pivot.to_numpy(),
        interpolation="nearest",
        vmin=-max_abs,
        vmax=max_abs,
        cmap="coolwarm",
    )

    plt.colorbar(
        label="Standardized meta-logit contribution"
    )

    plt.xticks(
        np.arange(
            len(
                pivot.columns
            )
        ),
        pivot.columns,
        rotation=40,
        ha="right",
    )

    plt.yticks(
        np.arange(
            len(
                pivot.index
            )
        ),
        pivot.index,
    )

    plt.title(
        "Case-Level Source Contributions Toward the Final Predicted Class"
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "case_level_source_contribution_heatmap.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

else:
    print(
        "Case-level meta-source contribution skipped "
        "because the saved meta-model could not be loaded."
    )

In [16]:
# ============================================================
# CELL 16 — GLOBAL SOURCE-IMPORTANCE FIGURE
# ============================================================

if IMPORTANCE_AVAILABLE:
    imp = (
        GLOBAL_SOURCE_IMPORTANCE_DF
        .sort_values(
            "normalized_meta_contribution",
            ascending=False,
        )
        .copy()
    )

    plt.figure(
        figsize=(9.0, 5.4)
    )

    plt.bar(
        imp[
            "source_model"
        ],
        100
        * imp[
            "normalized_meta_contribution"
        ],
    )

    plt.xticks(
        rotation=40,
        ha="right",
    )

    plt.ylabel(
        "Share of mean absolute meta-logit contribution (%)"
    )

    plt.title(
        "Global Source Contributions in the Frozen Logistic Stacker"
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "global_meta_source_importance.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

In [17]:
# ============================================================
# CELL 17 — FINAL XAI / INTERPRETABILITY AUDIT
# ============================================================

summary = {
    "protocol":
        "original stratified 80/10/10",

    "test_n":
        len(Y_TEST),

    "final_method":
        FINAL_CONFIG[
            "final_method"
        ],

    "test_used_for_model_selection":
        False,

    "gradcam": {
        "visualized_branches": [
            "SimCLR representative seed",
            "Rotation SSL fine-tuned branch",
        ],

        "simclr_representative_seed":
            int(
                SIMCLR_REP_SEED
            ),

        "simclr_seed_selection":
            "validation BACC, macro-F1, accuracy; seed tie-break",

        "rotation_checkpoint":
            str(
                ROTATION_WEIGHTS
            ),

        "target_layer":
            LAST_CONV_LAYER,

        "cam_target":
            "ground-truth class",

        "correct_per_class_cases":
            len(
                correct_case_indices
            ),

        "rescue_or_disagreement_cases":
            len(
                rescue_case_indices
            ),
    },

    "ensemble_interpretation": {
        "source_models":
            MODEL_NAMES,

        "anchor":
            ANCHOR_NAME,

        "stacking_alpha":
            BEST_ALPHA,

        "class_bias":
            [
                float(v)
                for v in CLASS_BIAS
            ],

        "meta_model_loaded":
            bool(
                IMPORTANCE_AVAILABLE
            ),

        "reconstruction_audit":
            RECONSTRUCTION_AUDIT,

        "leave_one_source_out_available":
            bool(
                len(
                    LOSO_DF
                ) > 0
            ),
    },

    "interpretation_rule": (
        "Grad-CAM is used only for the two foundational CNN branches. "
        "The final proposed method is an ensemble and therefore is not "
        "misrepresented as having a single Grad-CAM map. Its decision "
        "mechanism is instead explained through the frozen stacking "
        "meta-classifier's source-level contributions, anchor blend, "
        "and class-bias correction."
    ),
}

with open(
    AUDIT_DIR / "J7_xai_interpretability_summary.json",
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        default=float,
    )

print(
    json.dumps(
        summary,
        indent=2,
        default=float,
    )
)

{
  "protocol": "original stratified 80/10/10",
  "test_n": 2534,
  "final_method": "NoTTA_StackingBlend_ClassBias",
  "test_used_for_model_selection": false,
  "gradcam": {
    "visualized_branches": [
      "SimCLR representative seed",
      "Rotation SSL fine-tuned branch"
    ],
    "simclr_representative_seed": 62,
    "simclr_seed_selection": "validation BACC, macro-F1, accuracy; seed tie-break",
    "rotation_checkpoint": "/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/extended_imagenet_rotation_finetune/imagenet_rotation_finetune_best.weights.h5",
    "target_layer": "conv5_block3_out",
    "cam_target": "ground-truth class",
    "correct_per_class_cases": 8,
    "rescue_or_disagreement_cases": 8
  },
  "ensemble_interpretation": {
    "source_models": [
      "SimCLR_NoTTA",
      "SimCLR_SelectedTTA",
      "Rotation",
      "FusionConcat",
      "FusionGatedV2",
      "FusionGatedV3",
      "ResidualFusion"
    ],
    "anchor"

In [18]:
# ============================================================
# CELL 18 — COMPLETION MARKER + ZIP
# ============================================================

marker = {
    "stage":
        "J7_original_split_xai_ensemble_contribution",

    "status":
        "complete",

    "protocol":
        "original_stratified_80_10_10",

    "test_n":
        len(Y_TEST),

    "final_method":
        FINAL_CONFIG[
            "final_method"
        ],

    "simclr_representative_seed":
        int(
            SIMCLR_REP_SEED
        ),

    "correct_gradcam_cases":
        len(
            correct_case_indices
        ),

    "rescue_gradcam_cases":
        len(
            rescue_case_indices
        ),

    "meta_model_loaded":
        bool(
            IMPORTANCE_AVAILABLE
        ),

    "test_used_for_selection":
        False,

    "output_rel":
        "j7_original_split_xai",
}

MARKER_PATH = (
    WORKING
    / "J7_XAI_COMPLETE.json"
)

with open(
    MARKER_PATH,
    "w",
) as f:
    json.dump(
        marker,
        f,
        indent=2,
    )

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR
    / "J7_XAI_COMPLETE.json",
)

ZIP_PATH = (
    WORKING
    / "J7_ORIGINAL_SPLIT_XAI_RESULTS.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(
        OUTPUT_DIR.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    p.relative_to(
                        OUTPUT_DIR.parent
                    )
                ),
            )

required = [
    TABLE_DIR
    / "selected_xai_cases.csv",

    FIG_DIR
    / "gradcam_correct_per_class.png",

    FIG_DIR
    / "gradcam_final_rescue_cases.png",

    AUDIT_DIR
    / "ensemble_reconstruction_audit.json",

    AUDIT_DIR
    / "J7_xai_interpretability_summary.json",

    MARKER_PATH,

    ZIP_PATH,
]

if IMPORTANCE_AVAILABLE:
    required.extend([
        TABLE_DIR
        / "global_meta_source_importance.csv",

        FIG_DIR
        / "global_meta_source_importance.png",
    ])

missing = [
    str(p)
    for p in required
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing J7 outputs:\n"
        + "\n".join(
            missing
        )
    )

print("=" * 96)
print("J7 XAI / ENSEMBLE INTERPRETABILITY COMPLETE")
print("=" * 96)

print(
    "Representative SimCLR seed:",
    SIMCLR_REP_SEED
)

print(
    "Correct-class Grad-CAM cases:",
    len(
        correct_case_indices
    )
)

print(
    "Rescue/disagreement Grad-CAM cases:",
    len(
        rescue_case_indices
    )
)

print(
    "Meta-model loaded:",
    IMPORTANCE_AVAILABLE
)

print(
    "ZIP:",
    ZIP_PATH
)

print()
print(
    "Save this Kaggle notebook Version WITH OUTPUTS "
    "and send the executed notebook back."
)

J7 XAI / ENSEMBLE INTERPRETABILITY COMPLETE
Representative SimCLR seed: 62
Correct-class Grad-CAM cases: 8
Rescue/disagreement Grad-CAM cases: 8
Meta-model loaded: True
ZIP: /kaggle/working/J7_ORIGINAL_SPLIT_XAI_RESULTS.zip

Save this Kaggle notebook Version WITH OUTPUTS and send the executed notebook back.
